In [ ]:
"""
visualize_path.py

Plots the original swaths and the optimized route produced by
routing_engine.py, so you can visually sanity-check the solution.

- Solid colored segments = painting (in-swath, mandatory, free)
- Dashed gray arrows     = travel (in-air, the cost being minimized)
- Numbers                = order in which each swath is visited

Run this AFTER routing_engine.py has produced optimized_path.json.
"""

import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def main():
    swaths = load_json("generated_swaths.json")
    result = load_json("optimized_path.json")

    path = result["path"]          # flat list of [x, y] points, 2 per swath
    order = result["swath_order"]  # list of {'swath_idx': i, 'direction': 0/1}
    depot = result["depot"]

    fig, ax = plt.subplots(figsize=(9, 7))

    # Draw every swath faintly first, as a reference for the full canvas
    for s in swaths:
        x = [s["start"][0], s["end"][0]]
        y = [s["start"][1], s["end"][1]]
        ax.plot(x, y, color="#cccccc", linewidth=2, zorder=1)

    # Walk the solved route: pairs of points in `path` are (entry, exit) of
    # each visited swath, in order. Between consecutive swaths is travel.
    prev_exit = tuple(depot)
    for step, i in enumerate(range(0, len(path), 2)):
        entry = tuple(path[i])
        exit_ = tuple(path[i + 1])
        swath_number = step + 1

        # Travel segment (dashed arrow) from previous exit to this entry
        ax.annotate(
            "",
            xy=entry,
            xytext=prev_exit,
            arrowprops=dict(arrowstyle="->", color="gray", linestyle="dashed",
                             linewidth=1.2, alpha=0.8),
            zorder=2,
        )

        # Painting segment (solid, colored, thicker) from entry to exit
        ax.plot([entry[0], exit_[0]], [entry[1], exit_[1]],
                 color="tab:blue", linewidth=3.5, solid_capstyle="round", zorder=3)

        # Label the order at the entry point
        ax.annotate(str(swath_number), xy=entry, xytext=(4, 4),
                    textcoords="offset points", fontsize=9, fontweight="bold",
                    color="darkblue", zorder=4)

        prev_exit = exit_

    # Mark the depot
    ax.plot(depot[0], depot[1], marker="*", color="red", markersize=16, zorder=5)
    ax.annotate("depot", xy=depot, xytext=(6, -12), textcoords="offset points",
                color="red", fontsize=9)

    legend_handles = [
        mpatches.Patch(color="tab:blue", label="Painting (mandatory)"),
        mpatches.Patch(color="gray", label="Travel (minimized cost)"),
    ]
    ax.legend(handles=legend_handles, loc="upper right")

    ax.set_title(f"Optimized coverage path — total travel distance: "
                 f"{result['total_air_distance']:.3f}")
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.tight_layout()
    plt.savefig("route_visualization.png", dpi=150)
    print("Saved plot to route_visualization.png")
    plt.show()


if __name__ == "__main__":
    main()